# Volume Indicators (Category 1/5)

This notebook shows all `volume_*` indicators from `ta` on NSE/NIFTY OHLCV data.
Diagrams are shown inline only (no image files are written).

In [6]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ta import add_all_ta_features
from ta.utils import dropna

In [7]:
DATA_PATH = Path('NIFTY 50-02-04-2025-to-02-04-2026.csv')
CATEGORY_PREFIX = 'volume_'
MAX_INDICATORS = None  # Set an int to limit plots while exploring

COLUMN_ALIASES = {
    'Date': ['Date', 'DATE', 'Timestamp', 'Trading Date'],
    'Open': ['Open', 'OPEN', 'Open Price', 'OPEN_PRICE'],
    'High': ['High', 'HIGH', 'High Price', 'HIGH_PRICE'],
    'Low': ['Low', 'LOW', 'Low Price', 'LOW_PRICE'],
    'Close': ['Close', 'CLOSE', 'Close Price', 'CLOSE_PRICE', 'Prev Close'],
    'Volume': ['Volume', 'VOLUME', 'Shares Traded', 'Total Traded Quantity', 'TOTTRDQTY', 'Total Traded Volume'],
}


def _find_column(columns, candidates):
    normalized = {c.strip().lower(): c for c in columns}
    for candidate in candidates:
        match = normalized.get(candidate.strip().lower())
        if match:
            return match
    raise KeyError(f'Missing required column. Expected one of: {candidates}')


def _to_numeric(series):
    cleaned = series.astype(str).str.replace(',', '', regex=False).str.replace('%', '', regex=False).str.strip()
    return pd.to_numeric(cleaned, errors='coerce')


def _parse_dates(series):
    values = series.astype(str).str.strip()
    for fmt in ('%d-%b-%Y', '%d-%b-%y', '%d %b %Y', '%Y-%m-%d'):
        parsed = pd.to_datetime(values, format=fmt, errors='coerce')
        if parsed.notna().any():
            return parsed
    return pd.to_datetime(values, errors='coerce')


def load_nse_history(csv_path):
    raw = pd.read_csv(csv_path)
    normalized = pd.DataFrame()
    normalized['Date'] = _parse_dates(raw[_find_column(raw.columns, COLUMN_ALIASES['Date'])])
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        source_col = _find_column(raw.columns, COLUMN_ALIASES[col])
        normalized[col] = _to_numeric(raw[source_col])
    normalized = normalized.dropna(subset=['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    return normalized.sort_values('Date').reset_index(drop=True)

In [8]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'CSV not found: {DATA_PATH}. Update DATA_PATH in this notebook.')

df = dropna(load_nse_history(DATA_PATH))
df_ta = add_all_ta_features(df.copy(), open='Open', high='High', low='Low', close='Close', volume='Volume', fillna=False)

category_columns = [c for c in df_ta.columns if c.startswith(CATEGORY_PREFIX)]
if MAX_INDICATORS is not None:
    category_columns = category_columns[:MAX_INDICATORS]

print(f'Total {CATEGORY_PREFIX} indicators: {len(category_columns)}')
category_columns

Total volume_ indicators: 10


['volume_adi',
 'volume_obv',
 'volume_cmf',
 'volume_fi',
 'volume_em',
 'volume_sma_em',
 'volume_vpt',
 'volume_vwap',
 'volume_mfi',
 'volume_nvi']

## Interactive Plotly View

This final section renders interactive charts for the same category indicators using Plotly.

- Pan/zoom to inspect regions.
- Toggle traces from legend.
- No image files are written.

In [10]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError as exc:
    raise ImportError("Install plotly to use interactive charts: pip install plotly") from exc

MAX_INTERACTIVE_INDICATORS = None  # Set an int to limit interactive charts.
interactive_columns = category_columns[:MAX_INTERACTIVE_INDICATORS] if MAX_INTERACTIVE_INDICATORS else category_columns

for col in interactive_columns:
    plot_df = df_ta[['Date', 'Close', col]].dropna()
    if plot_df.empty:
        continue

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(f'NIFTY 50 Close | {col}', f'Volume Indicator | {col}')
    )
    fig.add_trace(
        go.Scatter(x=plot_df['Date'], y=plot_df['Close'], mode='lines', name='NIFTY 50 Close', line={'color': 'black'}),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=plot_df['Date'], y=plot_df[col], mode='lines', name=col),
        row=2,
        col=1,
    )
    fig.update_layout(height=680, template='plotly_white', showlegend=True)
    fig.show()